# Practice Assignment: Understanding Distributions Through Sampling

** *This assignment is optional, and I encourage you to share your solutions with me and your peers in the discussion forums!* **


To complete this assignment, create a code cell that:
* Creates a number of subplots using the `pyplot subplots` or `matplotlib gridspec` functionality.
* Creates an animation, pulling between 100 and 1000 samples from each of the random variables (`x1`, `x2`, `x3`, `x4`) for each plot and plotting this as we did in the lecture on animation.
* **Bonus:** Go above and beyond and "wow" your classmates (and me!) by looking into matplotlib widgets and adding a widget which allows for parameterization of the distributions behind the sampling animations.


Tips:
* Before you start, think about the different ways you can create this visualization to be as interesting and effective as possible.
* Take a look at the histograms below to get an idea of what the random variables look like, as well as their positioning with respect to one another. This is just a guide, so be creative in how you lay things out!
* Try to keep the length of your animation reasonable (roughly between 10 and 30 seconds).

In [2]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# generate 4 random variables from the random, gamma, exponential, and uniform distributions
x1 = np.random.normal(-2.5, 1, 10000)
x2 = np.random.gamma(2, 1.5, 10000)
x3 = np.random.exponential(2, 10000)+7
x4 = np.random.uniform(14,20, 10000)

# plot the histograms
plt.figure(figsize=(9,3))
plt.hist(x1, density=True, bins=20, alpha=0.5)
plt.hist(x2, density=True, bins=20, alpha=0.5)
plt.hist(x3, density=True, bins=20, alpha=0.5)
plt.hist(x4, density=True, bins=20, alpha=0.5);
plt.axis([-7,21,0,0.6])

plt.text(x1.mean()-1.5, 0.5, 'x1\nNormal')
plt.text(x2.mean()-1.5, 0.5, 'x2\nGamma')
plt.text(x3.mean()-1.5, 0.5, 'x3\nExponential')
plt.text(x4.mean()-1.5, 0.5, 'x4\nUniform');

RuntimeError: 'widget' is not a recognised GUI loop or backend name

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import ipywidgets as widgets
from IPython.display import display
import math

# 1. Define interactive widgets for distribution parameterization
style = {'description_width': 'initial'}
layout = widgets.Layout(width='95%')

# Normal parameters
norm_mean = widgets.FloatSlider(value=-2.5, min=-10.0, max=10.0, step=0.1, description='Normal Mean:', style=style, layout=layout)
norm_std = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Normal Std:', style=style, layout=layout)

# Gamma parameters
gamma_shape = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.1, description='Gamma Shape:', style=style, layout=layout)
gamma_scale = widgets.FloatSlider(value=1.5, min=0.1, max=5.0, step=0.1, description='Gamma Scale:', style=style, layout=layout)

# Exponential parameters
exp_scale = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='Exp Scale:', style=style, layout=layout)
exp_offset = widgets.FloatSlider(value=7.0, min=0.0, max=15.0, step=0.1, description='Exp Offset:', style=style, layout=layout)

# Uniform parameters
unif_low = widgets.FloatSlider(value=14.0, min=5.0, max=25.0, step=0.1, description='Uniform Low:', style=style, layout=layout)
unif_high = widgets.FloatSlider(value=20.0, min=10.0, max=30.0, step=0.1, description='Uniform High:', style=style, layout=layout)

# Animation parameters
num_samples = widgets.IntSlider(value=500, min=100, max=1000, step=50, description='Max Samples:', style=style, layout=layout)
anim_interval = widgets.IntSlider(value=50, min=20, max=200, step=10, description='Speed (ms):', style=style, layout=layout)

# 2. Setup the subplots figure
fig, axs = plt.subplots(2, 2, figsize=(9, 7))
axs = axs.ravel()
titles = ['Normal Distribution', 'Gamma Distribution', 'Exponential Distribution', 'Uniform Distribution']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']

# Global variables to store the data and animation reference
x1, x2, x3, x4 = None, None, None, None
anim = None

def init_datasets():
    global x1, x2, x3, x4
    n_max = num_samples.value
    
    # Generate maximum required samples based on current widget parameters
    x1 = np.random.normal(norm_mean.value, norm_std.value, n_max)
    x2 = np.random.gamma(gamma_shape.value, gamma_scale.value, n_max)
    x3 = np.random.exponential(exp_scale.value, n_max) + exp_offset.value
    
    # Ensure uniform low is strictly less than uniform high
    low = unif_low.value
    high = unif_high.value
    if high <= low:
        high = low + 0.1
    x4 = np.random.uniform(low, high, n_max)

# Initialize data initially
init_datasets()

# 3. Define the update function for the animation
def update(frame):
    if x1 is None:
        return
        
    for i in range(4):
        axs[i].clear()
        
        # Determine dynamic ranges based on current widget settings to keep the distributions perfectly in frame
        if i == 0:
            data = x1[:frame]
            mu, sigma = norm_mean.value, norm_std.value
            xlim = (mu - 4 * sigma, mu + 4 * sigma)
            ylim = (0, 1.2 / (sigma * np.sqrt(2 * np.pi)))
            bins = np.linspace(xlim[0], xlim[1], 30)
        elif i == 1:
            data = x2[:frame]
            shape, scale = gamma_shape.value, gamma_scale.value
            xlim = (0, shape * scale + 4 * np.sqrt(shape) * scale)
            # Calculate mode-based peak density for Gamma limits
            if shape > 1:
                peak = ((shape - 1)**(shape - 1) * np.exp(1 - shape)) / (scale * math.gamma(shape))
            else:
                peak = 1.0 / scale
            ylim = (0, max(peak * 1.2, 0.2))
            bins = np.linspace(xlim[0], xlim[1], 30)
        elif i == 2:
            data = x3[:frame]
            scale, offset = exp_scale.value, exp_offset.value
            xlim = (offset, offset + 4 * scale)
            ylim = (0, 1.2 / scale)
            bins = np.linspace(xlim[0], xlim[1], 30)
        else:
            data = x4[:frame]
            low, high = unif_low.value, unif_high.value
            if high <= low:
                high = low + 0.1
            xlim = (low - 0.5, high + 0.5)
            ylim = (0, 1.2 / (high - low))
            bins = np.linspace(low, high, 30)
            
        axs[i].hist(data, bins=bins, density=True, color=colors[i], alpha=0.6, edgecolor='black', linewidth=0.5)
        axs[i].set_xlim(xlim)
        axs[i].set_ylim(ylim)
        axs[i].set_title(f'{titles[i]} (n={frame})')
        axs[i].set_ylabel('Probability Density')
        axs[i].set_xlabel('Value')
        axs[i].grid(True, linestyle='--', alpha=0.3)
        
    fig.tight_layout()

# 4. Function to start/restart the animation
def start_animation():
    global anim
    if anim is not None:
        try:
            anim.event_source.stop()
        except:
            pass
            
    n_max = num_samples.value
    interval = anim_interval.value
    
    # Generate frame steps from 100 to N, scaling step size appropriately
    frame_steps = np.arange(100, n_max + 1, max(1, (n_max - 100) // 40))
    
    anim = animation.FuncAnimation(
        fig, 
        update, 
        frames=frame_steps, 
        interval=interval, 
        repeat=True, 
        repeat_delay=1000
    )

# Callback function when a slider changes
def on_parameter_change(change):
    # Enforce uniform constraint if needed
    if change['owner'] == unif_low and unif_high.value <= unif_low.value:
        unif_high.value = unif_low.value + 1.0
    elif change['owner'] == unif_high and unif_high.value <= unif_low.value:
        unif_low.value = max(0.0, unif_high.value - 1.0)
        
    init_datasets()
    start_animation()

# Bind observations to widgets
for slider in [norm_mean, norm_std, gamma_shape, gamma_scale, exp_scale, exp_offset, unif_low, unif_high, num_samples, anim_interval]:
    slider.observe(on_parameter_change, names='value')

# 5. Display interface layout
# Group layout by distribution columns
col1 = widgets.VBox([
    widgets.HTML('<b>Normal Parameters:</b>'), norm_mean, norm_std,
    widgets.HTML('<br><b>Exponential Parameters:</b>'), exp_scale, exp_offset
], layout=widgets.Layout(padding='10px', border='1px solid #ccc', margin='5px', flex='1'))

col2 = widgets.VBox([
    widgets.HTML('<b>Gamma Parameters:</b>'), gamma_shape, gamma_scale,
    widgets.HTML('<br><b>Uniform Parameters:</b>'), unif_low, unif_high
], layout=widgets.Layout(padding='10px', border='1px solid #ccc', margin='5px', flex='1'))

col3 = widgets.VBox([
    widgets.HTML('<b>Animation Settings:</b>'), num_samples, anim_interval
], layout=widgets.Layout(padding='10px', border='1px solid #ccc', margin='5px', flex='1'))

dashboard = widgets.HBox([col1, col2, col3], layout=widgets.Layout(width='100%'))

print("Interactive Distribution Animator Initialized!")
display(dashboard)
start_animation()


RuntimeError: 'widget' is not a recognised GUI loop or backend name